In [235]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
# input = r"/Users/cmdb/Quant_Bio_Project/Quant-Bio-Project/segmentation_test.tif" 
# img = cv.imread(input, cv.IMREAD_GRAYSCALE)



In [236]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

def gaussian_kernal(size,std):
    kernel = np.fromfunction(
        lambda x,y: np.divide(1,2*np.pi* std**2) * 
        np.exp(
            -((x-(size-1)/2)**2 + (y-(size-1)/2)**2) / (2* std**2)
               ),
        (size,size)
    )
    return np.array(kernel/np.sum(kernel))

def find(frame,mask,contour,pixelSize):

    """
    This function takes in an image and draws the given contours.  It returns an inverted boolean array
    where true = pixel within a contour. This gives the number of pixels within a given contour to calculate area, as well as the 
    starting point for blob labeling and tracking.
    """
    #dont want to edit the original image
    filled_mask = np.copy(mask)
    #draw the contour
    filled_mask = cv.drawContours(filled_mask,[contour], 0,(0, 255, 0),thickness=cv.FILLED)
    #boolean mask and returning it, ensuring true = 0, false = 1. The sum of this is pixel size. 
    image_mask = filled_mask > 0
    total_pixels = np.sum(~image_mask.astype(int))
    image_mask = ~image_mask
    total_pixels = np.sum(image_mask.astype(int))
    #output [frame, x,y, Area, TotalIntensity]
    total_intensity = np.sum(mask*(image_mask.astype(int)))
    ##Find X,Y center of mass based on contours. 
    #https://docs.opencv.org/3.4/dd/d49/tutorial_py_contour_features.html
    M = cv.moments(contour)
    x = int(M['m10']/M['m00'])
    y = int(M['m01']/M['m00'])
    #in units of pixelSize(micron squared)
    total_area = total_pixels*(pixelSize**2)
    return [frame,[x,y],total_area,total_intensity]
kernel = gaussian_kernal(3,np.sqrt(3))
low_thresh: int = 13
high_thresh: int = 50

#contour filtering
# we want anything equal to or greater than 5
filter_val = 10

##Pixel Size. This is used for area calculations. Where the area of a pixel is pixelSize**2
pixel_size: float = 0.115 


## Apply pixel normalization

Min Pixel Value = min(image)
Max Pixel Value = max(image)

Normalized = {pixel - min pixel}/{max pixel - min pixel}

Next? Check contrs output

In [237]:
# epsilon = cv.arcLength(cnt,True)
# approx = cv.approxPolyDP(cnt,epsilon,True)

## Time to work on the full stack analysis

#### Working with a stacked tiff

Implemented

## Convert from uint16 to uint8
CV2 threshold function only works with uint8 values, so we have to convert it. This is done by dividing it by 256 and assigning it as a uint8 datatype. 

## Histogram Normalization

In [238]:
# image1 = cv.filter2D(tiff_stack[10],-1,kernel)
# plt.imshow(image1)
# plt.show()

Converted stacked tiff into opencv 

In [239]:
import skimage.io as io
tiff_stack = io.imread("/Users/cmdb/Quant_Bio_Project/Quant-Bio-Project/cell2.tif",plugin='tifffile')
tiff_stack = (tiff_stack/256).astype(np.uint8)
copied = np.copy(tiff_stack)
frame_dict = {}
for i in range(len(tiff_stack)):
    ret,thresh = cv.threshold(tiff_stack[i,:,:],low_thresh,high_thresh,cv.THRESH_BINARY)
    contours,hierarchy = cv.findContours(thresh, 1, 2)
    #filtering out contours less than 10
    contours = [lst for lst in contours if len(lst) >= filter_val]
    frame_dict[f"Timepoint {i}"] = []
    for j in contours:
        #thresholding step needed to skip bad areas. Area == 0 is nothing. 
        area = cv.moments(j)["m00"]
        if area > 0:
            centroid_output = find(i,copied[i,:,:],j,pixel_size)
            frame_dict[f"Timepoint {i}"].append(centroid_output)


## Tracking by Elucidian Distance

In [241]:
##Challenges
# iterate through the dictionary by frame ##### success
# iterate through the dictionary by frame and frame + 1 ##### success
# pull x,y coords ##### success
# create blobs for n ##### success
# # pair n+1 with n blobs (first two frames only)

def elucidian_distance(n_coords,n1_coords):
   # based on 2d distance formula of sqrt((x2-x1)**2 + (y2-y1)**2))
   x1 = n_coords[0]
   y1 = n_coords[1]
   x2 = n1_coords[0]
   y2 = n1_coords[1]
   distance = np.sqrt((x2-x1)**2 + (y2-y1)**2)
   return distance

def search_matching_by_distance(blobs,n_coords,n1_coords,thresh):

   
   """
   take a single coord from n: this is the input n_coords <- incorrect, you might pair previous coords with n+1 coords. 
   iterate through all n+1, this is all n+1 coords
   find coords within the threshold distance
   if within threshold distance, assign n+1 to that blob
   if no blob is found within threshold distance
   if multiple blobs found within threshold distance
         find a way to score it?
         cross that bridge when we have to 
   create new blob 
   repeat 

   """
   #dictionary with current blobs
   blob_dict = blobs
   distances = []
   for i in n1_coords:
      #first [1] is coords
      #second [] is x,y where x = 0, y = 1
      distance = elucidian_distance(n_coords[1],i[1])
   pass

def blob_tracking_simple(frame,dict,n,n1,threshold):
   pairs = []
   #input n is all blobs in frame
   #n[i] denotes the individual blobs
   for i in range(len(n)):
      #this assigns the first frame centroids as blobs i+1. these are the first blobs 
      if frame == 0:
         dict[f"Blob {i+1}"] = [n[i]]
      #this function takes in a single blob from n and compares it to all n1 coords
      search_matching_by_distance(dict,n_coords=n[i],n1_coords=n1,thresh=threshold)
      for j in range(len(n1)):

         distance = elucidian_distance(n_coords=n[i][1],n1_coords=n1[j][1])
         if distance < threshold:
            pass
      print(i)


   pass
   
#keys list
list_of_keys = sorted(frame_dict.keys())
#this is where we will be storing the tracked blobs
blobs: dict = {}
for i in range(1):
   n = list_of_keys[i]
   n1 = list_of_keys[i+1]
   coords_n = frame_dict[n]
   coords_n1 = frame_dict[n1]
   
   blobs = blob_tracking_simple(i,blobs,coords_n,coords_n1,threshold=5)


   

0
1
2
3
4
5
6
7


## this is going to get really messy

heres the matrix implementation of the hungarian algorithm based on cost-value matrixes. based on the wikipedia walkthrough
https://en.wikipedia.org/wiki/Hungarian_algorithm#Matrix_interpretation

## Step 1, Create the cost matrix
Here the cost between n and n+1 is the distance. 

In [242]:
def test_one_zeros(matrix,xy):
        num_zeros = np.sum(matrix == 0, axis=xy)
        if max(num_zeros) > 1:
            return False
        else:
            return True
def subtract_minimum(matrix):
    backup_matrix = np.copy(matrix)
    cnt = 0
    for i in range(len(matrix)):
        min = np.min(matrix[i,:])
        matrix[i,:] =  matrix[i,:] - min
        cnt += 1
    if test_one_zeros(matrix,xy=0) == False:
        for k in range((np.shape(backup_matrix)[1])):
            min = np.min(backup_matrix[:,k])
            backup_matrix[:,k] = backup_matrix[:,k] - min
        return backup_matrix
    else:
        return matrix               
def matrix_generation(p,o):
    #where p = n
    #where o = n+1
    top_row = []
    side_row = []
    for i in p:
        top_row.append(i[1])
    for j in o:
        side_row.append(j[1])
    #time to create the matrix
    cost_matrix = []
    for b in top_row:
        output_row = []
        for c in side_row:
            cost = np.sqrt((c[0]-b[0])**2 + (c[1]-b[1])**2)
            output_row.append(cost)
        cost_matrix.append(output_row)

    return(np.array(cost_matrix))


                
for i in range(1):
   n = list_of_keys[i]
   n1 = list_of_keys[i+1]
   coords_n = frame_dict[n]
   coords_n1 = frame_dict[n1]
   cost_matrix_original = matrix_generation(p=coords_n,o=coords_n1)
cost_matrix = subtract_minimum(cost_matrix_original)
if test_one_zeros(cost_matrix,xy=0) == False:
    print('cols')
if test_one_zeros(cost_matrix,xy=1) == False:
    print('rows')
## Where there is a row and column with just a single zero, that is what should be assigned to a blob. 
## if there is a row or column with multiple zeros, that is the excess blob and should be added to the dictionary.
#I really hope this works 

def find_minimum_cost(matrix):
    null = np.where(cost_matrix ==0)
    return null
#where we have a cost value of 0
output = np.where(cost_matrix == 0)

"""
    Step 2

    Subtract the minimum from each row in the cost matrix. Each row should have 1 zero
"""

rows


'\n    Step 2\n\n    Subtract the minimum from each row in the cost matrix. Each row should have 1 zero\n'

In [243]:
def flatten_coords(list_of_lists):
    return [sublist[1] for sublist in list_of_lists if len(sublist) > 1]

In [244]:
for i in range(1):
   n = list_of_keys[i]
   n1 = list_of_keys[i+1]
   coords_n = flatten_coords(frame_dict[n]) #current
   coords_n1 = flatten_coords(frame_dict[n1]) #next frame
   cost_matrix = cdist(coords_n,coords_n1)
   rows = cost_matrix.min(axis=1).argsort()
   cols = cost_matrix.argmin(axis=1)[rows]
   

In [245]:
class Objects:
    def flatten_coords(list_of_lists):
        return [sublist[1] for sublist in list_of_lists if len(sublist) > 1]
    def initial_blobs(self):
        list_frames = list(frame_dict.keys())
        first_frame = self.frames[list_frames[0]]
        for i in range(len(first_frame)):
            self.blobs[f"Blob {self.num_blobs + 1}"] = first_frame[i]
            self.num_blobs = len(self.blobs.keys())
        self.current_blobs.append(list(self.blobs.keys()))
        return
            
    def __init__(self,frame_dict):
        #this will contain all the blobs
        self.frames = frame_dict
        num_frames = len(frame_dict.keys())
        self.blobs: dict = {}
        self.num_blobs = len(self.blobs.keys())
        self.current_blobs = []
        self.initial_blobs()
    #new input is going to add blobs based on the cost matrix
    def elucidian_distance(self,n_coords,n1_coords):
        # based on 2d distance formula of sqrt((x2-x1)**2 + (y2-y1)**2))
        x1 = n_coords[0]
        y1 = n_coords[1]
        x2 = n1_coords[0]
        y2 = n1_coords[1]
        distance = np.sqrt((x2-x1)**2 + (y2-y1)**2)
        return distance
    def distance_thresholding(self,distance,threshold):
        if distance > threshold:
            return False
        else:
            return True

    def pairing(self,matrix,coords_n,coords_n1):
        rows = matrix.min(axis=1).argsort()
        cols = matrix.argmin(axis=1)[rows]
        usedRows = set()
        usedCols = set()
        for row,col in zip(rows,cols):
            #skips previously used rows/columns
            if row in usedRows or col in usedCols:
                continue
            n= (coords_n[row])
            n1 = (coords_n1[cols[col]])
            for blob in self.blobs:
                distance = elucidian_distance(n,n1)
                if self.distance_thresholding(distance,15) == True and self.blobs[blob]:
                    pass






   
            
    def new_input(self,n,n1):
        coords_n = flatten_coords(self.frames[n]) #current
        coords_n1 = flatten_coords(self.frames[n1]) #next frame
        cost_matrix = cdist(coords_n,coords_n1)
        self.pairing(cost_matrix,coords_n,coords_n1)
        
        
        
        
        pass
"""
coords_n[rows[i]] accesses the i-th point from coords_n based on the sorted order of minimum distances.
coords_n1[cols[i]] accesses the closest point in coords_n1 to the point coords_n[rows[i]]

"""  

'\ncoords_n[rows[i]] accesses the i-th point from coords_n based on the sorted order of minimum distances.\ncoords_n1[cols[i]] accesses the closest point in coords_n1 to the point coords_n[rows[i]]\n\n'

In [246]:
blobs = Objects(frame_dict)
keys = list(blobs.frames.keys())
for i in range(len(keys) -1):
    blobs.new_input(n=keys[i],n1=keys[i+1])

    

IndexError: index 8 is out of bounds for axis 0 with size 7

In [ ]:
def pair_blobs(row,col,n,n1,blob_dict):

rows gives the sorted order of points in coords_n based on their minimum distance to points in coords_n1.
cols gives the indices of the closest points in coords_n1 to each point in coords_n, but in the order determined by rows. This is essentially the optimal matching of points between the two sets based on the minimum cost (distance).

In [ ]:
usedRows = set()
usedCols = set()

In [ ]:
else:
			# grab the set of object IDs and corresponding centroids
			objectIDs = list(self.objects.keys())
			objectCentroids = list(self.objects.values())
			# compute the distance between each pair of object
			# centroids and input centroids, respectively -- our
			# goal will be to match an input centroid to an existing
			# object centroid
			D = dist.cdist(np.array(objectCentroids), inputCentroids)
			# in order to perform this matching we must (1) find the
			# smallest value in each row and then (2) sort the row
			# indexes based on their minimum values so that the row
			# with the smallest value is at the *front* of the index
			# list
			rows = D.min(axis=1).argsort()
			# next, we perform a similar process on the columns by
			# finding the smallest value in each column and then
			# sorting using the previously computed row index list
			cols = D.argmin(axis=1)[rows]

## step 2

Now we need to sort the blobs.  This part will be difficult

1) blobs need to be paired based on cost
2) some frames will have unmatched blobs
    1) for isntance, frame 1 has 6 blobs and frame 2 has 7. How do you determine the extra? 
        i) distance, rows, etc. Distance threshold will probably need to be determined

In [ ]:

## remember row is a blob
## 0 and 0 are together
## 1 and 1
## 3 an 2
## 4 and 3
## 5 and 5



### take unique counts of blobs, add on.




def find_duplicates(input_list):
    seen = set()  
    duplicates = []  

    for item in input_list:
        if item in seen:
            duplicates.append(item)
        else:
            seen.add(item)

    return duplicates

def find_pairs(output):
    #this will give the duplicate rows in the where.output 
    dup_n1 = find_duplicates(output[0])
    dup_n = find_duplicates(output[1])
    pairs = []
    for k in len(output[0]):
        pairs.append([output[0][k],output[1][k]])
    return pairs


def blob_pairing(blob_dict, where_output,n,n1,cost_matrix_original,threshold):
    ## the goal of this function is the pair the output of np.where and the cost analysis matrix implementation
    #reminders
    ## rows are n+1, cols are n
    ## rows that appear more than once, are new blobs, add them to blob_dict
    ## must recalculate matrix without that row then pair the blobs!
    #this will add blob pairs that are within the length of n


    #pairs are found as coords [n1,n]
    # filter pairs that are thresholding
    #
    #find pairs is going to take in the indexes that have a 1:1 match for zero. It will update the inputted list while also
    #outputting a new list with the mismatches 2:1 or 1:2 pairs
    pairs = find_pairs(where_output)
    for p in pairs:
        if cost_matrix_original[p[0],p[1]] > threshold:
            threshold_trigger = True
            continue
        

    for j in range(len(n)):
        n1_index = where_output[0][j]
        n_index = where_output[1][j]
        threshold_trigger = False
        #here we only add blob pairs if the distance is under the thresholding
        if cost_matrix_original[n1_index,n_index] > threshold:
            threshold_trigger = True
            continue
        if str(n[n_index]) not in blob_dict.keys() and threshold_trigger == False:
            blob_dict[str(n[n_index])] = f"Blob {len(set(blob_dict.values()))+1}"
        #this adds n1 only if n is present and the threshold trigger has not been tripped. 
        if (str(n[n_index])) in blob_dict.keys() and threshold_trigger == False:
            blob_dict[str(n1[n1_index])] = blob_dict[str(n[n_index])]

    return blob_dict

    
tracking_dict = {}
tracking_dict = blob_pairing(tracking_dict,output,coords_n,coords_n1,cost_matrix_original,threshold=20)

In [ ]:
print((output[1]))

## msd calculation function

In [ ]:
def msd_fft(r):
                """This is function that calculates the MSD. This was copied from the stack exchange link below, so all credit goes to that post
                
                https://stackoverflow.com/questions/34222272/computing-mean-square-displacement-using-python-and-fft
                Args:
                    r (array): input array in x,y(,z) 

                Returns:
                    array: Returns the MSD values for that track. 
                """
                def autocorrFFT(x):
                        N=len(x)
                        F = np.fft.fft(x, n=2*N)  #2*N because of zero-padding
                        PSD = F * F.conjugate()
                        res = np.fft.ifft(PSD)
                        res= (res[:N]).real   #now we have the autocorrelation in convention B
                        n=N*np.ones(N)-np.arange(0,N) #divide res(m) by (N-m)
                        return res/n #this is the autocorrelation in convention A
                N=len(r)
                D=np.square(r).sum(axis=1) 
                D=np.append(D,0) 
                S2=sum([autocorrFFT(r[:, i]) for i in range(r.shape[1])])
                Q=2*D.sum()
                S1=np.zeros(N)
                for m in range(N):
                        Q=Q-D[m-1]-D[N-m]
                        S1[m]=Q/(N-m)
                return S1-2*S2  

In [ ]:
## This is the MSD computation per track and subseuqnet weighting/updates

In [ ]:
def msd_compute(self):
                """MSD Computation based on msdanalyzer computeMSD function.  msd.fft calculation is pulled from stack exchange(see that function for the link)
                """
                coords_list = [[np.array([x, y]) for x, y in zip(self.x[idx], self.y[idx])] for idx in range(self.ntracks)]
                msd_fast = {}
                tolerance = 12
                frame_data = self.frames
                max_spots = self.__max_spots              
                #update coords list
                for i in range(self.ntracks):
                        weights = np.arange(1,len(coords_list[i])+1)
                        reversed_weights = weights[::-1]
                        results = (self.__msd_fft(np.array(coords_list[i])))
                        #setting t0 to 0 do 
                        results[0] = 0
                        prenan = np.empty((max_spots - len(coords_list[i])))
                        prenan[:] = np.nan
                        zers = np.zeros((max_spots - len(coords_list[i])))
                        # c is for weights
                        #b is results 
                        c = np.concatenate([reversed_weights,zers])
                        b = np.concatenate([results,prenan])
                        msd_fast[i] = [self.timepoints,b,c]
                        #updating self
                        self.msd.append(b)
                        self.msd_weights.append(c)     
            
                return

In [ ]:
def calc_diffusion(self, dimension = None, clipfactor = None,filter = False,filter_r2 = None):
                num_filtered = 0
                self.ntracks = len(self.msd)
                removed_indices = []
                if clipfactor == None:
                        clip = 0.25
                else: 
                        if clipfactor > 1:
                                clip = 1
                        if clipfactor < 0:
                                clip = 0
                        if clipfactor <= 1 or clipfactor >= 0:
                                clip = clipfactor
                if filter == True and filter_r2 != None: 
                        filter_val_r2 = filter_r2
                if filter == True and filter_r2 == None:
                        filter_val_r2 = 0.8
                if dimension == None:
                        n = 2
                if dimension != None:
                        n = dimension
                # MSD = 2nDt
                time = self.timepoints
               
                coef = []
                inter = []
                #same idea for loglog, just dont take the log of the values
                print("Calculating diffusion coeffcient in uM^2/s through the weighted linear fit of " + str(self.ntracks) + " MSD curves.")
                print("Only taking the first " + str(int(clip*100)) + " percent of each MSD curve. ") 
                if filter_r2 == True:
                        print("Filtering out tracks that have an r2 value below " + str(filter_val_r2))
                for i in range(len(self.msd)):
                        #REMOVE LATER   
                        msds= self.msd[i]
                        valid = ~np.isnan(msds)
                        y = self.msd[i][valid]
                        x = time[valid].reshape((-1,1))
                        weights = self.msd_weights[i][valid]
                        length = int(np.round(len(y)*clip))
                        x_axis = x[:length]
                        y_axis = y[:length]
                        if x_axis.shape == (0,) or y_axis.shape == (0,) or len(x_axis) < 5:
                                num_filtered += 1
                                removed_indices.append(False)
                                continue
                        model = LinearRegression().fit(x_axis,y_axis,sample_weight=weights[:length])
                        # Diffusion_Coefficient = MSD/2n
                        r2 = model.score(x_axis,y_axis)

                        if filter == True and (r2 < filter_val_r2 or r2 == np.nan) :
                                removed_indices.append(False)
                                num_filtered += 1
                                continue
                        coef.append(model.coef_)
                        inter.append(model.intercept_)
                        diff = np.divide(model.coef_, 2*n)
                        removed_indices.append(True)
                        self.diff_coef.append(float(diff[0]))
                        self.diff_r2.append(r2)
                if filter == True:
                        print("Total tracks filtered = " + str(int(num_filtered)))
                        
                self.removal_report[f'Total tracks removed during diffusion calculations: '] = num_filtered
                
                self.__update(removed_indices)
                self.loglog = np.array(self.loglog, dtype=object)
                self.loglog_r2 = np.array(self.loglog_r2, dtype=object)
                
                self.loglog = self.loglog[removed_indices]
                self.loglog_r2 = self.loglog_r2[removed_indices]
                
                self.intercept = inter
                self.coef = coef
                print("\n")
        """
        Optional Analysis below
        - Viscosity
        - Direction analysis
        - Displacement
        """